# 강의 03 · 실습 3 — RAG 검색기 구축 · (6) 고난도 III

## 1. 문제상황

- 구름월드 고객센터 검색기는 질문에 가까운 FAQ 청크(chunk)를 찾아 주지만, 손님에게는 청크 그대로가 아니라 답 문장이 나가야 합니다.
- 답에는 어느 FAQ 항목(행 번호와 카테고리)을 근거로 했는지 출처가 붙어야, 담당자가 나중에 확인할 수 있습니다.
- 시험 운영에서 임계값 1.5는 「표 물리면 얼마 돌려줘요」에 점수 1.47의 분실물 청크를 통과시켜 엉뚱한 근거가 붙었습니다. 담당자는 임계값을 1.3으로 낮추기로 했습니다.
- 그러면 「표 물리면 얼마 돌려줘요」처럼 표현이 달라 못 찾은 질문이 컷되므로, 컷된 질문을 바로 모른다고 하지 말고 질의를 한 번 바꿔 다시 검색한 뒤에도 근거가 없을 때만 모른다고 답하기를 원합니다.

## 2. 문제와 목표

- **문제**: 검색 결과가 청크로 끝나 답 문장과 출처가 없고, 컷된 질문을 표현 차이인지 문서 밖인지 가리지 않고 모두 모른다고 처리합니다.
- **목표**: 상위 청크를 프롬프트에 동봉해 모델이 근거로만 답 문장을 만들고 출처(행 번호·카테고리)를 병기하며, 컷된 질문은 모델에게 질의를 한 번 다시 쓰게 해 재검색하고 그래도 컷이면 고정 안내 문장으로 보내는 프로그램을 만듭니다.
    - FAQ 파일: `day05_faq_구름월드.csv`(32행). 저장 디렉터리: `chroma_db`.
    - 임계값: 1.3. 재검색: 한 번.
    - 답 생성과 질의 재작성에 쓰는 모델: `openai/gpt-5.6-luna`(litellm 경유).
    - 고정 안내 문장: 「문서에서 근거를 찾지 못했습니다. 안내 창구로 문의해 주세요.」 출처 형식: `[행 N · 카테고리]`.
    - 질문 세 개: 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**: 문서 안 질문의 답에 FAQ 내용과 출처(행 번호와 카테고리)가 붙고, 표현이 다른 질문(「표 물리면 얼마 돌려줘요?」)은 재작성 후 통과해 답이 나오며, 문서 밖 질문은 재작성 후에도 컷되어 고정 안내 문장이 나오는 것을 출력에서 확인합니다.

## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항

## 5. 코드 골격

## 6. 코드 — 스텝바이스텝

### 단계 0 — 주어진 것

라이브러리 불러오기, `.env` 읽기, 모델 준비는 주어진 것입니다. 아래 셀을 고치지 않고 그대로 실행합니다.

- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, `OPENAI_API_KEY=발급받은_키` 한 줄만 넣습니다.


In [ ]:
import csv
import os

from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

# 주어진 자료 — 값과 이름을 그대로 씁니다.
QUESTIONS = [
    "자유이용권 환불 규정 알려 주세요",
    "표 물리면 얼마 돌려줘요?",
    "파이썬 리스트 정렬은 어떻게 하나요?",
]


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 문서 안 질문은 첫 검색에서 통과하고, 답 끝에 행 번호와 카테고리 출처가 붙습니다.
2. 표현이 다른 질문은 첫 검색에서 컷되고, 다시 쓴 질의로 통과해 환불 규정 답과 출처가 나옵니다.
3. 문서 밖 질문은 다시 쓴 질의로도 컷되어 고정 안내 문장이 나옵니다.

세 가지가 모두 확인되면 완성입니다.